In [0]:
#SILVER TO GOLD
import yaml
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType, DecimalType, IntegerType
from pyspark.sql.functions import col, when, lit, sum, coalesce, count, regexp_replace, monotonically_increasing_id, date_format, from_utc_timestamp, current_timestamp, year, month, dayofmonth, quarter, current_date, min as spark_min, sequence, explode, to_date
from delta.tables import DeltaTable

dbutils.widgets.text("NB_Storage_Account","")
storage_account = dbutils.widgets.get("NB_Storage_Account")

dbutils.widgets.text("NB_Year", "")
dbutils.widgets.text("NB_Month", "")
param_year = dbutils.widgets.get("NB_Year")
param_month = dbutils.widgets.get("NB_Month")
print(param_year)
print(param_month)

def table_has_schema(table_name):
    try:
        df = spark.table(table_name)
        return len(df.columns) > 0
    except Exception:
        return False

### DIMENSION DATE FOR 3 TABLES(ORDER,PAYMENT,INVOICE) ###

In [0]:
df_all_dates = spark.sql("SELECT sequence(to_date('2023-01-01'), to_date('2030-12-31'), interval 1 day) as date_array") \
    .select(explode(col("date_array")).alias("TransactionDate"))

df_dim_date = (
    df_all_dates
    .withColumn("DateKey", date_format(col("TransactionDate"), "yyyyMMdd").cast("int"))
    .withColumn("Year", year(col("TransactionDate")).cast("string"))
    .withColumn("Month", month(col("TransactionDate")).cast("string"))
    .withColumn("Day", dayofmonth(col("TransactionDate")).cast("string"))
    .withColumn("MonthName", date_format(col("TransactionDate"), "MMMM"))
    .withColumn("Quarter", quarter(col("TransactionDate")))
    .withColumn("DayName", date_format(col("TransactionDate"), "EEEE"))
    .select(
        "DateKey",
        "TransactionDate",
        "Year",
        "Month",
        "Day",
        "MonthName",
        "Quarter",
        "DayName"
    )
)

if spark.catalog.tableExists("<catalog>.gold.dim_date"):
    (DeltaTable.forName(spark, "<catalog>.gold.dim_date").alias("target")
        .merge(df_dim_date.alias("source"), "target.DateKey = source.DateKey")
        .whenNotMatchedInsertAll()
        .execute())
    print("dim_date: added successfully")
else:
    df_dim_date.write \
        .format("delta") \
        .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_date/") \
        .saveAsTable("<catalog>.gold.dim_date")
    print("dim_date: created and added successfully")


### DIMENSION & FACT ORDER ###

In [0]:
if not table_has_schema("<catalog>.silver.dtl_order"):
    print(f"SKIPPED: Silver dtl_order Year: {param_year} Month: {param_month} table does not exist")
else:
    df_dtl_order = spark.table("<catalog>.silver.dtl_order").filter(col("Year")==param_year).filter(col("Month")==param_month)
    if df_dtl_order.isEmpty():
        print(f"No Data Silver dtl_order Year: {param_year} Month: {param_month}")
    else:
        #DIM ITEM
        df_source_orderitem = df_dtl_order.groupBy("ItemId", "ClassName").agg(spark_min("TransactionDate").alias("StartDate"))

        if spark.catalog.tableExists("<catalog>.gold.dim_item"):

            df_sink_orderitem = spark.table("<catalog>.gold.dim_item") \
                .filter(col("IsActive") == True).select("ItemKey","ItemID", col("ClassName").alias("ClassName_Old"))

            df_compare_orderitem = df_source_orderitem.join(df_sink_orderitem, on="ItemID", how = "left") \
                .select(df_source_orderitem["ItemID"], df_source_orderitem["ClassName"], df_source_orderitem["StartDate"], df_sink_orderitem["ItemKey"], df_sink_orderitem["ClassName_Old"])

            df_filter_new_orderitem = df_compare_orderitem.filter(col("ItemKey").isNull()).select("ItemID","ClassName", "StartDate")
            df_filter_changed_orderitem = df_compare_orderitem.filter(col("ItemKey").isNotNull() & (col("ClassName") != col("ClassName_Old")))

            max_value_orderitem = (spark.sql("select max(ItemKey) from <catalog>.gold.dim_item").collect()[0][0] or 0) + 1

            df_newversions_orderitem = (
                df_filter_new_orderitem.select("ItemID","ClassName","StartDate") \
                .unionByName(df_filter_changed_orderitem.select("ItemID","ClassName","StartDate")) \
                .coalesce(1) \
                .withColumn("ItemKey", max_value_orderitem + monotonically_increasing_id()) \
                .withColumn("EndDate", lit(None).cast("date")) \
                .withColumn("IsActive", lit(True))
            )

            delta_tbl_orderitem = DeltaTable.forName(spark, "<catalog>.gold.dim_item")

            df_expire_orderitem = df_filter_changed_orderitem.select("ItemKey", col("StartDate").alias("ChangeDate"))
            (
                delta_tbl_orderitem.alias("target") \
                .merge(df_expire_orderitem.alias("source"), "target.ItemKey = source.ItemKey") \
                .whenMatchedUpdate(set={
                    "EndDate": col("source.ChangeDate"),
                    "IsActive": lit(False)
                }) \
                    .execute()
            )

            df_existing_active_orderitem = spark.table("<catalog>.gold.dim_item") \
                .filter(col("IsActive") == True) \
                .select("ItemId", "ClassName")
            
            df_newversions_orderitem = df_newversions_orderitem.join(
                df_existing_active_orderitem,
                on=["ItemId", "ClassName"],
                how="left_anti"
            )

            df_newversions_orderitem.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_item")
            print(f"Saved Gold.dim_item Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_orderitem = (
                df_source_orderitem \
                .withColumn("ItemKey", monotonically_increasing_id() + 1) \
                .withColumn("EndDate", lit(None).cast("date")) \
                .withColumn("IsActive", lit(True))
            )
            df_first_orderitem.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_item/") \
                .saveAsTable("<catalog>.gold.dim_item")
            print(f"Saved Gold.dim_item Data For: Year={param_year}, Month={param_month}")

        #DIM ORDER STATUS

        df_source_orderstatus = df_dtl_order.select("Status").distinct()

        if spark.catalog.tableExists("<catalog>.gold.dim_orderstatus"):

            df_sink_orderstatus = spark.table("<catalog>.gold.dim_orderstatus") \
                .select("StatusKey","Status")
            df_compare_orderstatus = df_source_orderstatus.join(df_sink_orderstatus, on="Status", how = "left")
            df_filter_new_orderstatus = df_compare_orderstatus.filter(col("StatusKey").isNull()).select("Status")
            
            max_value_orderstatus = (spark.sql("select max(StatusKey) from <catalog>.gold.dim_orderstatus").collect()[0][0] or 0) + 1
            df_newversions_orderstatus = (
                df_filter_new_orderstatus \
                .withColumn("StatusKey", max_value_orderstatus + monotonically_increasing_id())
            )

            df_newversions_orderstatus.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_orderstatus")
            print(f"Saved Gold.dim_orderstatus Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_orderstatus = df_source_orderstatus \
                .withColumn("StatusKey", monotonically_increasing_id() + 1)
            df_first_orderstatus.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_orderstatus/") \
                .saveAsTable("<catalog>.gold.dim_orderstatus")
            print(f"Saved Gold.dim_orderstatus Data For: Year={param_year}, Month={param_month}")

        #DIM ORDERTRANSACTION TYPE
        df_source_ordertransactiontype = df_dtl_order.select("TransactionType").distinct()

        if spark.catalog.tableExists("<catalog>.gold.dim_ordertransactiontype"):
            df_sink_ordertransactiontype = spark.table("<catalog>.gold.dim_ordertransactiontype") \
                .select("TransactionKey","TransactionType")
            df_compare_ordertransactiontype = df_source_ordertransactiontype.join(df_sink_ordertransactiontype, on="TransactionType", how = "left")
            df_filter_new_ordertransactiontype = df_compare_ordertransactiontype.filter(col("TransactionKey").isNull()).select("TransactionType")

            max_value_ordertransactiontype = (spark.sql("select max(TransactionKey) from <catalog>.gold.dim_ordertransactiontype").collect()[0][0] or 0) + 1
            df_newversions_ordertransactiontype = (
                df_filter_new_ordertransactiontype \
                .withColumn("TransactionKey", max_value_ordertransactiontype + monotonically_increasing_id())
            )

            df_newversions_ordertransactiontype.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_ordertransactiontype")
            print(f"Saved Gold.dim_ordertransactiontype Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_ordertransactiontype = df_source_ordertransactiontype \
                .withColumn("TransactionKey", monotonically_increasing_id() + 1)
            df_first_ordertransactiontype.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_ordertransactiontype/") \
                .saveAsTable("<catalog>.gold.dim_ordertransactiontype")
            print(f"Saved Gold.dim_ordertransactiontype Data For: Year={param_year}, Month={param_month}")

        #FACT ORDER
        df_get_dim_date = spark.table("<catalog>.gold.dim_date") \
            .select("DateKey", "TransactionDate")

        df_get_dim_item = spark.table("<catalog>.gold.dim_item") \
            .filter(col("IsActive") == True) \
            .select("ItemKey", "ItemID","StartDate", "EndDate")

        df_get_dim_orderstatus = spark.table("<catalog>.gold.dim_orderstatus") \
            .select("StatusKey", "Status")

        df_get_dim_ordertransactiontype = spark.table("<catalog>.gold.dim_ordertransactiontype") \
            .select("TransactionKey", "TransactionType")

        df_fact_order = df_dtl_order.alias("o") \
            .join(df_get_dim_date.alias("d"), col("o.TransactionDate").cast("date") == col("d.TransactionDate").cast("date"), how = "left") \
            .join(df_get_dim_item.alias("i"), col("o.ItemId") == col("i.ItemId"), how = "left") \
            .join(df_get_dim_orderstatus.alias("s"), col("o.Status") == col("s.Status"), how = "left") \
            .join(df_get_dim_ordertransactiontype.alias("t"), col("o.TransactionType") == col("t.TransactionType"), how = "left") \
            .select(col("d.DateKey"),
                    col("i.ItemKey"),
                    col("s.StatusKey"),
                    col("t.TransactionKey"),
                    col("o.Net"),
                    col("o.Tax"),
                    col("o.AdjustedNet"),
                    col("o.AdjustedTax"),
                    col("o.Discount"),
                    col("o.Year"),
                    col("o.Month"),
                    col("o.Day")
                    )
        if spark.catalog.tableExists("<catalog>.gold.fact_order"):
            df_fact_order.write \
                .format("delta") \
                .mode("overwrite") \
                .option("replaceWhere", f"Year = '{param_year}' AND Month = '{param_month}'") \
                .saveAsTable("<catalog>.gold.fact_order")
            print(f"Saved Gold.fact_order Data For: Year={param_year}, Month={param_month}")
        else:
            df_fact_order.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/fact_order/") \
                .partitionBy("Year", "Month", "Day") \
                .saveAsTable("<catalog>.gold.fact_order")
            print(f"Saved Gold.fact_order Data For: Year={param_year}, Month={param_month}")

### DIMENSION & FACT PAYMENT ###

In [0]:
if not table_has_schema("<catalog>.silver.dtl_payment"):
    print(f"SKIPPED: Silver dtl_payment Year: {param_year} Month: {param_month} table does not exist")
else:
    df_dtl_payment = spark.table("<catalog>.silver.dtl_payment").filter(col("Year")==param_year).filter(col("Month")==param_month)
    if df_dtl_payment.isEmpty():
        print(f"No Data Silver dtl_payment Year: {param_year} Month: {param_month}")
    else:
        #DIM PAYMENT TENDER ID
        df_source_paymenttender = df_dtl_payment.groupBy("TenderId", "TenderType").agg(spark_min("TransactionDate").alias("StartDate"))

        if spark.catalog.tableExists("<catalog>.gold.dim_tender"):
            df_sink_paymenttender = spark.table("<catalog>.gold.dim_tender") \
                .filter(col("IsActive") == True) \
                .select("TenderKey","TenderId", col("TenderType").alias("TenderType_Old"))

            df_compare_paymenttender = df_source_paymenttender.join(df_sink_paymenttender, on="TenderId", how="left") \
                    .select(df_source_paymenttender["TenderId"], df_source_paymenttender["TenderType"], df_source_paymenttender["StartDate"], df_sink_paymenttender["TenderKey"], df_sink_paymenttender["TenderType_Old"])

            df_filter_new_paymenttender = df_compare_paymenttender.filter(col("TenderKey").isNull()).select("TenderId","TenderType","StartDate")
            df_filter_changed_paymenttender = df_compare_paymenttender.filter(col("TenderKey").isNotNull() & (col("TenderType") != col("TenderType_Old")))

            max_value_paymenttender = (spark.sql("select max(TenderKey) from <catalog>.gold.dim_tender").collect()[0][0] or 0 ) + 1

            df_newversions_paymenttender = (
                df_filter_new_paymenttender.select("TenderId", "TenderType", "StartDate") \
                .unionByName(df_filter_changed_paymenttender.select("TenderId", "TenderType", "StartDate")) \
                .coalesce(1) \
                .withColumn("TenderKey", max_value_paymenttender + monotonically_increasing_id()) \
                .withColumn("EndDate", lit(None).cast("date")) \
                .withColumn("IsActive", lit(True))
            )

            delta_tbl_paymenttender = DeltaTable.forName(spark, "<catalog>.gold.dim_tender")

            df_expire_paymenttender = df_filter_changed_paymenttender.select("TenderKey", col("StartDate").alias("ChangedDate"))
            (
                delta_tbl_paymenttender.alias("target") \
                .merge(df_expire_paymenttender.alias("source"), "target.TenderKey = source.TenderKey") \
                .whenMatchedUpdate(set={
                    "EndDate": col("source.ChangedDate"),
                    "IsActive": lit(False)
                })
                .execute()
            )

            df_existing_active_paymenttender = spark.table("<catalog>.gold.dim_tender") \
                .filter(col("IsActive") == True) \
                .select("TenderId", "TenderType")
            
            df_newversions_paymenttender = df_newversions_paymenttender.join(
                df_existing_active_paymenttender,
                on=["TenderId", "TenderType"],
                how="left_anti"
            )


            df_newversions_paymenttender.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_tender")
            print(f"Saved Gold.dim_tender Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_paymenttender = (
                df_source_paymenttender \
                .withColumn("TenderKey", monotonically_increasing_id() + 1) \
                .withColumn("EndDate", lit(None).cast("date")) \
                .withColumn("IsActive", lit(True))
            )
            df_first_paymenttender.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_tender/") \
                .saveAsTable("<catalog>.gold.dim_tender")
            print(f"Saved Gold.dim_tender Data For: Year={param_year}, Month={param_month}")

        #DIM PAYMENT STATUS
        df_source_paymentstatus = df_dtl_payment.select("Status").distinct()

        if spark.catalog.tableExists("<catalog>.gold.dim_paymentstatus"):
            df_sink_paymentstatus = spark.table("<catalog>.gold.dim_paymentstatus") \
                    .select("StatusKey", "Status")
            df_compare_paymentstatus = df_source_paymentstatus.join(df_sink_paymentstatus, on="Status", how="left")
            df_filter_new_paymentstatus = df_compare_paymentstatus.filter(col("StatusKey").isNull()).select("Status")

            max_value_paymentstatus = (spark.sql("select max(StatusKey) from <catalog>.gold.dim_paymentstatus").collect()[0][0] or 0 ) + 1
            df_newversions_paymentstatus = (
                df_filter_new_paymentstatus \
                .withColumn("StatusKey", max_value_paymentstatus + monotonically_increasing_id())
            )

            df_newversions_paymentstatus.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_paymentstatus")
            print(f"Saved Gold.dim_paymentstatus Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_paymentstatus = df_source_paymentstatus \
                .withColumn("StatusKey", monotonically_increasing_id() + 1)
            df_first_paymentstatus.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_paymentstatus/") \
                .saveAsTable("<catalog>.gold.dim_paymentstatus")
            print(f"Saved Gold.dim_paymentstatus Data For: Year={param_year}, Month={param_month}")

        #FACT PAYMENT
        df_get_dim_date = spark.table("<catalog>.gold.dim_date") \
            .select("DateKey", "TransactionDate")

        df_get_dim_tender = spark.table("<catalog>.gold.dim_tender") \
            .filter(col("IsActive") == True) \
            .select("TenderKey", "TenderId", "StartDate", "EndDate")

        df_get_dim_paymentstatus = spark.table("<catalog>.gold.dim_paymentstatus") \
            .select("StatusKey", "Status")

        df_fact_payment = df_dtl_payment.alias("p") \
            .join(df_get_dim_date.alias("d"), col("p.TransactionDate").cast("date") == col("d.TransactionDate").cast("date"), how="left") \
            .join(df_get_dim_tender.alias("t"), col("p.TenderId") == col("t.TenderId"), how="left") \
            .join(df_get_dim_paymentstatus.alias("s"), col("p.Status") == col("s.Status"), how="left") \
            .select(col("d.DateKey"),
                    col("t.TenderKey"),
                    col("s.StatusKey"),
                    col("p.Amount"),
                    col("p.Year"),
                    col("p.Month"),
                    col("p.Day")
            )
        if spark.catalog.tableExists("<catalog>.gold.fact_payment"):
            df_fact_payment.write \
                .format("delta") \
                .mode("overwrite") \
                .option("replaceWhere", f"Year = '{param_year}' AND Month = '{param_month}'") \
                .saveAsTable("<catalog>.gold.fact_payment")
            print(f"Saved Gold.fact_payment Data For: Year={param_year}, Month={param_month}")
        else:
            df_fact_payment.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/fact_payment/") \
                .partitionBy("Year", "Month", "Day") \
                .saveAsTable("<catalog>.gold.fact_payment")
            print(f"Saved Gold.fact_payment Data For: Year={param_year}, Month={param_month}")

### DIMENSION & FACT INVOICE ###

In [0]:
if not table_has_schema("<catalog>.silver.hdr_invoice"):
    print(f"SKIPPED: Silver hdr_invoice Year: {param_year} Month: {param_month} table does not exist")
else:
    df_hdr_invoice = spark.table("<catalog>.silver.hdr_invoice").filter(col("Year")==param_year).filter(col("Month")==param_month)
    if df_hdr_invoice.isEmpty():
        print(f"No Data Silver hdr_invoice Year: {param_year} Month: {param_month}")
    else:
        #DIM INVOICE STATUS
        df_source_invoicestatus = df_hdr_invoice.select("Status").distinct()

        if spark.catalog.tableExists("<catalog>.gold.dim_invoicestatus"):
            df_sink_invoicestatus = spark.table("<catalog>.gold.dim_invoicestatus") \
                .select("StatusKey", "Status")
            df_compare_invoicestatus = df_source_invoicestatus.join(df_sink_invoicestatus, on="Status", how="left")
            df_filter_new_invoicestatus = df_compare_invoicestatus.filter(col("StatusKey").isNull()).select("Status")

            max_value_invoicestatus = (spark.sql("select max(StatusKey) from <catalog>.gold.dim_invoicestatus").collect()[0][0] or 0 ) + 1
            df_newversions_invoicestatus = (
                df_filter_new_invoicestatus \
                    .withColumn("StatusKey", max_value_invoicestatus + monotonically_increasing_id())
            )

            df_newversions_invoicestatus.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_invoicestatus")
            print(f"Saved Gold.dim_invoicestatus Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_invoicestatus = df_source_invoicestatus \
                .withColumn("StatusKey", monotonically_increasing_id() + 1)
            
            df_first_invoicestatus.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_invoicestatus/") \
                .saveAsTable("<catalog>.gold.dim_invoicestatus")
            print(f"Saved Gold.dim_invoicestatus Data For: Year={param_year}, Month={param_month}")

        #DIM INVOICE TRANSACTIONTYPE
        df_source_invoicetransactiontype = df_hdr_invoice.select("TransactionType").distinct()

        if spark.catalog.tableExists("<catalog>.gold.dim_invoicetransactiontype"):
            df_sink_invoicetransactiontype = spark.table("<catalog>.gold.dim_invoicetransactiontype") \
                .select("TransactionKey", "TransactionType")

            df_compare_invoicetransactiontype = df_source_invoicetransactiontype.join(df_sink_invoicetransactiontype, on="TransactionType", how="left")
            df_filter_new_invoicetransactiontype = df_compare_invoicetransactiontype.filter(col("TransactionKey").isNull()).select("TransactionType")

            max_value_invoicetransactiontype = (spark.sql("select max(TransactionKey) from <catalog>.gold.dim_invoicetransactiontype").collect()[0][0] or 0 ) + 1
            df_newversions_invoicetransactiontype = (
                df_filter_new_invoicetransactiontype \
                .withColumn("TransactionKey", max_value_invoicetransactiontype + monotonically_increasing_id())
            )

            df_newversions_invoicetransactiontype.write.format("delta").mode("append").saveAsTable("<catalog>.gold.dim_invoicetransactiontype")
            print(f"Saved Gold.dim_invoicetransactiontype Data For: Year={param_year}, Month={param_month}")
        else:
            df_first_invoicetransactiontype = df_source_invoicetransactiontype \
                .withColumn("TransactionKey", monotonically_increasing_id() + 1)
            
            df_first_invoicetransactiontype.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/dim_invoicetransactiontype/") \
                .saveAsTable("<catalog>.gold.dim_invoicetransactiontype")
            print(f"Saved Gold.dim_invoicetransactiontype Data For: Year={param_year}, Month={param_month}")

        #FACT INVOICE
        df_get_dim_date = spark.table("<catalog>.gold.dim_date") \
            .select("DateKey", "TransactionDate")
        df_get_dim_invoicestatus = spark.table("<catalog>.gold.dim_invoicestatus") \
            .select("StatusKey", "Status")
        df_get_dim_invoicetransactiontype = spark.table("<catalog>.gold.dim_invoicetransactiontype") \
            .select("TransactionKey", "TransactionType")

        df_fact_invoice = df_hdr_invoice.alias("h") \
            .join(df_get_dim_date.alias("d"), col("h.TransactionDate").cast("date") == col("d.TransactionDate").cast("date"), how="left") \
            .join(df_get_dim_invoicestatus.alias("s"), col("h.Status") == col("s.Status"), how="left") \
            .join(df_get_dim_invoicetransactiontype.alias("t"), col("h.TransactionType") == col("t.TransactionType"), how="left") \
            .select(col("d.DateKey"),
                    col("s.StatusKey"),
                    col("t.TransactionKey"),
                    col("h.VatableSales"),
                    col("h.NonVatableSales"),
                    col("h.VatExemptSales"),
                    col("h.Vat"),
                    col("h.Discount"),
                    col("h.AmountDue"),
                    col("h.SeniorCitizenCount"),
                    col("h.GuestCount"),
                    col("h.Year"),
                    col("h.Month"),
                    col("h.Day")
            )
        if spark.catalog.tableExists("<catalog>.gold.fact_invoice"):
            df_fact_invoice.write \
                .format("delta") \
                .mode("overwrite") \
                .option("replaceWhere", f"Year = '{param_year}' AND Month = '{param_month}'") \
                .saveAsTable("<catalog>.gold.fact_invoice")
            print(f"Saved Gold.fact_invoice Data For: Year={param_year}, Month={param_month}")
        else:
            df_fact_invoice.write \
                .format("delta") \
                .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/sales/fact_invoice/") \
                .partitionBy("Year", "Month", "Day") \
                .saveAsTable("<catalog>.gold.fact_invoice")
            print(f"Saved Gold.fact_invoice Data For: Year={param_year}, Month={param_month}")